In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc


In [ ]:
# =========================
# INPUT
# =========================
H5AD = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/05b_trajectory_revisit/adata_CD8_subset_clusters_0_1_2_4.h5ad"

# =========================
# OUTPUT DIR (DE_DIR)
# =========================
# 你后续 driver gene 脚本用的是 DE_DIR 里这些 CSV
DE_DIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells"
os.makedirs(DE_DIR, exist_ok=True)

CLUSTER_COL = 'leiden_T_0.6_relabel'   # e.g. "leiden" or "cluster" or "leiden_cd8"
RESPONSE_COL = 'Respond'  # e.g. "Responder" or "response" or "RNR" or "is_responder"

print("DE_DIR:", DE_DIR)

In [ ]:
ad = sc.read_h5ad(H5AD)
print(ad)

In [ ]:
old_key = "leiden_T_0.6"
new_key = "leiden_T_0.6_relabel"

# 映射：old -> new
mapping = {
    "4": "1",
    "0": "2",
    "1": "3",
    "2": "4",
    }

In [ ]:
# 确保是字符串（scanpy 的 leiden 通常本来就是字符串）
ad.obs[old_key] = ad.obs[old_key].astype(str)

# 生成新标签；未在 mapping 里的（比如 7/8）会变成 NaN
ad.obs[new_key] = ad.obs[old_key].map(mapping)

# 可选：把没映射到的保留原值（例如 7/8），避免 NaN
ad.obs[new_key] = ad.obs[new_key].fillna(ad.obs[old_key])

In [ ]:
print(ad.obs[new_key].value_counts())

In [ ]:
ROOT_CLUSTER = "1"
CLUSTER_KEY = "leiden_T_0.6_relabel"
ROOT_STRATEGY = "top_naive"
NAIVE_KEY = "score_Naive"
NAIVE_TOP_FRAC = 0.10       # 前10%
NAIVE_MIN_CAND = 30         # 候选太少时的保底（按你数据量可改）


In [ ]:
from scipy import sparse
NAIVE_GENES = [
    "TCF7", "LEF1", "CCR7", "IL7R", "LTB", "MAL", "SELL",
    "TRAC", "CD3D", "CD3E"
]

genes = [g for g in NAIVE_GENES if g in ad.var_names]

# 1) 构建 log1p(scvi_norm_expr) layer（安全处理 sparse）
X = ad.layers["norm_expr"]
if sparse.issparse(X):
    X_log = X.copy()
    X_log.data = np.log1p(X_log.data)
else:
    X_log = np.log1p(X)

ad.layers["norm_expr_log1p"] = X_log

# 2) 用这个 layer 算 score
Xbak = ad.X
ad.X = ad.layers["norm_expr_log1p"]
sc.tl.score_genes(ad, gene_list=genes, score_name=NAIVE_KEY, use_raw=False)
ad.X = Xbak

In [ ]:
def pick_root_cell(ad: sc.AnnData, rep: str) -> int:
    """Pick a robust root cell (iroot) using a cluster-defined candidate set."""
    mask_root = (ad.obs[CLUSTER_KEY].astype(str) == str(ROOT_CLUSTER)).values
    idxs = np.where(mask_root)[0]
    if len(idxs) == 0:
        raise ValueError(f"No cells found in ROOT_CLUSTER={ROOT_CLUSTER} for {CLUSTER_KEY}.")

    # ----- candidate selection: top naive within root cluster -----
    if ROOT_STRATEGY == "top_naive":
        if NAIVE_KEY not in ad.obs.columns:
            raise ValueError(f"ROOT_STRATEGY=top_naive but {NAIVE_KEY} not in ad.obs.")
        sub = ad.obs.loc[mask_root, NAIVE_KEY].astype(float)

        # take top 10% (or at least NAIVE_MIN_CAND)
        n_cand = max(int(np.ceil(len(sub) * NAIVE_TOP_FRAC)), NAIVE_MIN_CAND)
        n_cand = min(n_cand, len(sub))
        cand_names = sub.nlargest(n_cand).index
        cand_idxs = np.where(ad.obs_names.isin(cand_names))[0]

        print(f"[INFO] Root candidates: top {n_cand}/{len(sub)} ({NAIVE_TOP_FRAC:.0%}) by {NAIVE_KEY} in cluster {ROOT_CLUSTER}.")
    else:
        # fallback: all cells in root cluster
        cand_idxs = idxs
        print(f"[INFO] Root candidates: all {len(cand_idxs)} cells in cluster {ROOT_CLUSTER}.")

    # ----- pick representation space for medoid -----
    X = ad.obsm[rep][cand_idxs]

    # ----- medoid: most central cell among candidates -----
    x2 = np.sum(X * X, axis=1, keepdims=True)
    D2 = x2 + x2.T - 2 * (X @ X.T)
    D2 = np.maximum(D2, 0.0)
    medoid_local = int(np.argmin(D2.mean(axis=1)))
    iroot = int(cand_idxs[medoid_local])

    print(f"[INFO] Root cell selected by medoid in {rep}: {ad.obs_names[iroot]} (iroot={iroot})")
    return iroot
sc.pp.neighbors(ad, use_rep="X_scVI", n_neighbors=50)
#ensure_neighbors_for_dpt(ad)
iroot = pick_root_cell(ad, rep="X_scVI")
ad.uns["iroot"] = iroot

# Diffmap + DPT
sc.tl.diffmap(ad)
sc.tl.dpt(ad)  # uses ad.uns["iroot"]
if "dpt_pseudotime" not in ad.obs:
    raise RuntimeError("scanpy did not create ad.obs['dpt_pseudotime'].")

In [ ]:
sc.pl.umap(ad, color='dpt_pseudotime', show=True)